# Detecting protest events on my data

In [ ]:
import sys

sys.path.append("../..")

In [15]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [13]:
from protest_impact.data.protests.detection import load_glpn_dataset

glpn = load_glpn_dataset()
glpn = glpn.rename_column("excerpt", "text")

Using custom data configuration default-3a509930dabb81fc


Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

0 tables [00:00, ? tables/s]

0 tables [00:00, ? tables/s]

0 tables [00:00, ? tables/s]

0 tables [00:00, ? tables/s]

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/usr/local/lib/python3.9/dist-packages/IPython/core/interactiveshell.py", line 3398, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_33/1996737641.py", line 5, in <cell line: 5>
    glpn = load_glpn_dataset()
  File "/notebooks/protest_impact/data/protests/detection/glpn.py", line 21, in load_glpn_dataset
    glpn = load_dataset("csv", data_files=data_files)
  File "/usr/local/lib/python3.9/dist-packages/datasets/load.py", line 1679, in load_dataset
  File "/usr/local/lib/python3.9/dist-packages/datasets/builder.py", line 704, in download_and_prepare
    ```py
  File "/usr/local/lib/python3.9/dist-packages/datasets/builder.py", line 793, in _download_and_prepare
    if is_local:  # if cache dir is local, check for available space
  File "/usr/local/lib/python3.9/dist-packages/datasets/builder.py", line 1271, in _prepare_split
    to write the train data, then `_generate_examples(file='test_data.zip')` t

In [14]:
from protest_impact.data.protests.detection import load_aglpn_dataset

protest_news = load_aglpn_dataset()
protest_news, protest_news["train"].features

Using custom data configuration protest-c0ba145d55e29542
Reusing dataset json (/root/.cache/huggingface/datasets/json/protest-c0ba145d55e29542/0.0.0/da492aad5680612e4028e7f6ddc04b1dfcec4b64db470ed7cc5f2bb265b9b6b5)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?ba/s]

AttributeError: module 'pyarrow._compute' has no attribute 'ScalarUdfContext'

In [ ]:
len(glpn["train"]), len(protest_news["train"])

NameError: name 'glpn' is not defined

In [ ]:
!python -m spacy download de_core_news_sm

In [6]:
import spacy

from protest_impact.data.protests.config import search_regex

nlp = spacy.load("de_core_news_sm", disable=["parser", "tagger", "ner", "tokenizer"])
nlp.add_pipe("sentencizer")


def kwic(text, n=0):
    sents = list(nlp(text).sents)
    kwics_nrs = set()
    for i, sent in enumerate(sents):
        if search_regex.search(sent.text_with_ws):
            kwics_nrs.add(i)
            for j in range(1, n + 1):
                kwics_nrs.add(i - j)
                kwics_nrs.add(i + j)
    kwic_text = ""
    for kwic_nr in sorted(list(kwics_nrs)):
        if kwic_nr >= 0 and kwic_nr < len(sents):
            if kwic_nr - 1 not in kwics_nrs:
                kwic_text += "\n...\n"
            kwic_text += sents[kwic_nr].text_with_ws
    return kwic_text

In [7]:
def kwic_dataset(dataset, n=0):
    return dataset.map(
        lambda x: {
            "text": x["meta"]["title"]
            + "\n\n"
            + kwic("\n".join(list(x["text"].split("\n"))[1:]), n=n)
        }
    )

In [8]:
protest_news = kwic_dataset(protest_news, n=2)

  0%|          | 0/575 [00:00<?, ?ex/s]

  0%|          | 0/115 [00:00<?, ?ex/s]

  0%|          | 0/460 [00:00<?, ?ex/s]

In [ ]:
if device == "cpu":
    n = 20
    glpn["train"] = glpn["train"].shuffle(seed=20230206).select(range(n))
    glpn["dev"] = glpn["dev"].shuffle(seed=20230206).select(range(n))
    glpn["test"] = glpn["test"].shuffle(seed=20230206).select(range(n))
    glpn["test.time"] = glpn["test.time"].shuffle(seed=20230206).select(range(n))
    glpn["test.loc"] = glpn["test.loc"].shuffle(seed=20230206).select(range(n))

    protest_news["train"] = (
        protest_news["train"].shuffle(seed=20230206).select(range(n))
    )
    protest_news["dev"] = protest_news["dev"].shuffle(seed=20230206).select(range(n))
    protest_news["test"] = protest_news["test"].shuffle(seed=20230206).select(range(n))

In [9]:
import evaluate
import numpy as np

metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [20]:
!ls models/deepset-gelectra-large-glpn/model

config.json  pytorch_model.bin	training_args.bin


In [10]:
from pathlib import Path

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


def train_model(model, tokenizer, description, dataset):
    model_location = Path("models") / description / "model"

    if model_location.exists():
        return AutoModelForSequenceClassification.from_pretrained(model_location)

    def tokenize_function(examples):
        return tokenizer(
            examples["text"], padding="max_length", truncation=True, max_length=512
        )

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    training_args = TrainingArguments(
        output_dir=model_location.parent / "checkpoints",
        evaluation_strategy="epoch",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        learning_rate=5e-6,
        weight_decay=0.2,
        num_train_epochs=6,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["dev"],
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(model_location)
    return model

In [11]:
# model_name = "deepset/gelectra-base"
model_name = "deepset/gelectra-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_vanilla = AutoModelForSequenceClassification.from_pretrained(model_name).to(
    device
)

Some weights of the model checkpoint at deepset/gelectra-large were not used when initializing ElectraForSequenceClassification: ['discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight']
- This IS expected if you are initializing ElectraForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at deepset/gelectra-large and are newly initialized: ['classifier.dense.weight', 'classif

In [12]:
model_glpn = train_model(
    model_vanilla, tokenizer, f"{model_name.replace('/', '-')}-glpn", glpn
)

Epoch,Training Loss,Validation Loss,F1
1,No log,0.625413,0.865497
2,0.505100,0.522486,0.889590
3,0.272800,0.647826,0.893082
4,0.160500,0.750222,0.880795
5,0.094000,0.773199,0.882353
6,0.059500,0.769343,0.891720


In [13]:
from evaluate import TextClassificationEvaluator


def evaluate(model, tokenizer, test_set):
    eval_results = TextClassificationEvaluator().compute(
        model_or_pipeline=model,
        data=test_set,
        input_column="text",
        label_column="label",
        label_mapping={"LABEL_0": 0, "LABEL_1": 1},
        # label_mapping={"irrelevant": 0, "relevant": 1},
        # label_mapping={"LABEL_0": "irrelevant", "LABEL_1": "relevant"},
        tokenizer=tokenizer,
        metric=metric,
    )
    return eval_results

In [46]:
from sklearn.metrics import classification_report
from transformers import pipeline


def evaluate_detail(model, tokenizer, test_set):
    classifier = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device,
        padding="max_length",
        truncation=True,
        max_length=512,
    )
    predictions = classifier(a["text"] for a in test_set)
    y_pred = [np.argmax(a["score"]) for a in predictions]
    y_true = [a["label"][-1] for a in test_set]
    print(sum(y_true) / len(y_true))
    print(sum(y_pred) / len(y_pred))
    print(classification_report(y_true, y_pred))

In [ ]:
[a["text"] for a in glpn["test"]]

: 

In [20]:
evaluate(model_glpn, tokenizer, glpn["test"])

{'f1': 0.7131782945736435,
 'total_time_in_seconds': 16.236549190012738,
 'samples_per_second': 33.68942461840753,
 'latency_in_seconds': 0.02968290528338709}

In [47]:
evaluate_detail(model_glpn, tokenizer, glpn["test"])

0.603290676416819
0.0
              precision    recall  f1-score   support

           0       0.40      1.00      0.57       217
           1       0.00      0.00      0.00       330

    accuracy                           0.40       547
   macro avg       0.20      0.50      0.28       547
weighted avg       0.16      0.40      0.23       547



/usr/local/lib/python3.9/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.9/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.9/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [14]:
evaluate(model_glpn, tokenizer, protest_news["test"])

Disabling tokenizer parallelism, we're using DataLoader multithreading already


{'f1': 0.5781990521327014,
 'total_time_in_seconds': 9.911018010927364,
 'samples_per_second': 46.41299203500875,
 'latency_in_seconds': 0.021545691328102964}

In [ ]:
evaluate_detail(model_glpn, tokenizer, protest_news["test"])

In [15]:
model_protest_news = train_model(
    model_vanilla,
    tokenizer,
    f"{model_name.replace('/', '-')}-protest-news",
    protest_news,
)

Epoch,Training Loss,Validation Loss,F1
1,No log,0.578986,0.681818
2,No log,0.272684,0.787879
3,No log,0.482148,0.727273
4,0.295200,0.412240,0.742857
5,0.295200,0.412495,0.742857
6,0.295200,0.438645,0.742857


In [17]:
evaluate(model_protest_news, tokenizer, protest_news["test"])

{'f1': 0.7910447761194029,
 'total_time_in_seconds': 10.377355182077736,
 'samples_per_second': 44.32728685960806,
 'latency_in_seconds': 0.022559467787125514}

In [ ]:
evaluate_detail(model_protest_news, tokenizer, protest_news["test"])

In [21]:
model_glpn_protest_news = train_model(
    model_glpn,
    tokenizer,
    f"{model_name.replace('/', '-')}-glpn-protest-news",
    protest_news,
)

Epoch,Training Loss,Validation Loss,F1
1,No log,0.619752,0.702703
2,No log,0.537081,0.702703
3,No log,0.444293,0.736842
4,0.021600,0.435692,0.800000
5,0.021600,0.545525,0.736842
6,0.021600,0.536098,0.756757


In [22]:
evaluate(model_glpn_protest_news, tokenizer, protest_news["test"])

{'f1': 0.7826086956521738,
 'total_time_in_seconds': 10.425367980031297,
 'samples_per_second': 44.12314278796508,
 'latency_in_seconds': 0.022663843434850645}

In [ ]:
evaluate_detail(model_protest_news, tokenizer, protest_news["test"])

Make predictions for annotating more relevant articles: